In [0]:
# ===========================================
# Notebook Name:
# 01_validate_archetypes
#
# Purpose:
# Sanity-check the deck archetype clustering
# produced by 00_build_deck_archetypes before
# the weekly ML pipeline is considered
# successful: cluster size balance,
# silhouette score quality, representative
# Pokemon per cluster, and a comparison
# against the previous run's metrics.
#
# Input:
# pokemon.gold.deck_archetypes
# pokemon.gold.deck_pokemon_features
# pokemon.ops.pipeline_state
#
# Output:
# None (validation only; raises on failure)
# ===========================================

In [0]:
import json

from pyspark.sql import functions as F

DECK_ARCHETYPES_TABLE = (
    "pokemon.gold.deck_archetypes"
)
DECK_POKEMON_FEATURES_TABLE = (
    "pokemon.gold.deck_pokemon_features"
)
PIPELINE_STATE_TABLE = (
    "pokemon.ops.pipeline_state"
)

PIPELINE_NAME = "weekly_ml_pipeline"
STAGE_NAME = "build_deck_archetypes"

MIN_SILHOUETTE_SCORE = 0.0
MAX_CLUSTER_SHARE = 0.7
TOP_POKEMON_PER_CLUSTER = 5

print("Input :", DECK_ARCHETYPES_TABLE)
print("Input :", DECK_POKEMON_FEATURES_TABLE)
print("Input :", PIPELINE_STATE_TABLE)

In [0]:
deck_archetypes_df = spark.table(
    DECK_ARCHETYPES_TABLE
)

deck_count = (
    deck_archetypes_df
    .select("deck_hash")
    .distinct()
    .count()
)

print("Archetyped deck count:", deck_count)

if deck_count == 0:
    raise ValueError(
        f"{DECK_ARCHETYPES_TABLE} is empty. "
        "Run 00_build_deck_archetypes first."
    )

In [0]:
cluster_sizes_df = (
    deck_archetypes_df
    .groupBy("cluster_id")
    .agg(
        F.countDistinct(
            "deck_hash"
        ).alias("cluster_size")
    )
    .withColumn(
        "cluster_share",
        F.round(
            F.col("cluster_size")
            / F.lit(deck_count),
            4,
        ),
    )
    .orderBy(
        F.col("cluster_size").desc()
    )
)

display(cluster_sizes_df)

singleton_cluster_count = (
    cluster_sizes_df
    .filter(F.col("cluster_size") == 1)
    .count()
)

print(
    "Singleton clusters "
    "(informational only):",
    singleton_cluster_count,
)

In [0]:
largest_cluster_share = (
    cluster_sizes_df
    .agg(
        F.max("cluster_share").alias(
            "max_share"
        )
    )
    .first()["max_share"]
)

print(
    "Largest cluster share:",
    largest_cluster_share,
)

if largest_cluster_share > MAX_CLUSTER_SHARE:
    raise ValueError(
        "Largest cluster holds "
        f"{largest_cluster_share:.0%} of all "
        f"decks, above the "
        f"{MAX_CLUSTER_SHARE:.0%} threshold. "
        "Clustering did not meaningfully "
        "separate archetypes."
    )

print(
    "Validation passed: "
    "no single cluster dominates"
)

In [0]:
model_summary_row = (
    deck_archetypes_df
    .select(
        "model_name",
        "cluster_count",
        "silhouette_score",
    )
    .distinct()
    .collect()
)

if len(model_summary_row) != 1:
    display(
        spark.createDataFrame(
            model_summary_row
        )
    )

    raise ValueError(
        "Expected a single "
        "(model_name, cluster_count, "
        "silhouette_score) combination for "
        "the latest run, found "
        f"{len(model_summary_row)}"
    )

current_model_name = (
    model_summary_row[0]["model_name"]
)
current_cluster_count = (
    model_summary_row[0]["cluster_count"]
)
current_silhouette_score = (
    model_summary_row[0][
        "silhouette_score"
    ]
)

print("Model name       :", current_model_name)
print("Cluster count    :", current_cluster_count)
print(
    "Silhouette score  :",
    current_silhouette_score,
)

In [0]:
if (
    current_silhouette_score is None
    or current_silhouette_score < -1.0
    or current_silhouette_score > 1.0
):
    raise ValueError(
        "Silhouette score "
        f"{current_silhouette_score} is "
        "outside the valid [-1, 1] range"
    )

if current_silhouette_score < MIN_SILHOUETTE_SCORE:
    raise ValueError(
        "Silhouette score "
        f"{current_silhouette_score} is below "
        f"the minimum quality threshold "
        f"{MIN_SILHOUETTE_SCORE}"
    )

print(
    "Validation passed: "
    "silhouette score is within range and "
    "meets the minimum quality threshold"
)

In [0]:
cluster_pokemon_usage_df = (
    spark.table(DECK_POKEMON_FEATURES_TABLE)
    .join(
        deck_archetypes_df.select(
            "deck_hash",
            "cluster_id",
        ),
        on="deck_hash",
        how="inner",
    )
    .groupBy("cluster_id", "card_name")
    .agg(
        F.countDistinct(
            "deck_hash"
        ).alias("decks_using_pokemon")
    )
    .join(
        cluster_sizes_df.select(
            "cluster_id",
            "cluster_size",
        ),
        on="cluster_id",
        how="left",
    )
    .withColumn(
        "inclusion_rate",
        F.round(
            F.col("decks_using_pokemon")
            / F.col("cluster_size"),
            4,
        ),
    )
)

clusters_without_pokemon = (
    cluster_sizes_df
    .join(
        cluster_pokemon_usage_df.select(
            "cluster_id"
        ).distinct(),
        on="cluster_id",
        how="left_anti",
    )
)

if clusters_without_pokemon.count() > 0:
    display(clusters_without_pokemon)

    raise ValueError(
        "Cluster(s) with no matching Pokemon "
        "features found. Check the join "
        "against "
        f"{DECK_POKEMON_FEATURES_TABLE}."
    )

print(
    "Validation passed: "
    "every cluster has representative "
    "Pokemon usage data"
)

In [0]:
from pyspark.sql.window import Window

representative_pokemon_window = (
    Window
    .partitionBy("cluster_id")
    .orderBy(
        F.col("inclusion_rate").desc(),
        F.col("decks_using_pokemon").desc(),
    )
)

representative_pokemon_df = (
    cluster_pokemon_usage_df
    .withColumn(
        "rank_in_cluster",
        F.row_number().over(
            representative_pokemon_window
        ),
    )
    .filter(
        F.col("rank_in_cluster")
        <= TOP_POKEMON_PER_CLUSTER
    )
    .orderBy(
        "cluster_id",
        "rank_in_cluster",
    )
)

display(representative_pokemon_df)

In [0]:
if spark.catalog.tableExists(
    PIPELINE_STATE_TABLE
):
    previous_state_row = (
        spark.table(PIPELINE_STATE_TABLE)
        .filter(
            (F.col("pipeline_name") == PIPELINE_NAME)
            & (F.col("stage_name") == STAGE_NAME)
        )
        .select("metrics_json")
        .first()
    )

else:
    previous_state_row = None

if (
    previous_state_row is None
    or previous_state_row["metrics_json"] is None
):
    print(
        "No previous run recorded for "
        f"{PIPELINE_NAME}/{STAGE_NAME}. "
        "Skipping comparison "
        "(this looks like the first run)."
    )

else:
    previous_metrics = json.loads(
        previous_state_row["metrics_json"]
    )

    previous_cluster_count = (
        previous_metrics.get("cluster_count")
    )
    previous_silhouette_score = (
        previous_metrics.get(
            "silhouette_score"
        )
    )
    previous_deck_count = (
        previous_metrics.get("deck_count")
    )

    print(
        "Previous cluster_count  :",
        previous_cluster_count,
    )
    print(
        "Current  cluster_count  :",
        current_cluster_count,
    )
    print(
        "Previous silhouette_score:",
        previous_silhouette_score,
    )
    print(
        "Current  silhouette_score:",
        current_silhouette_score,
    )
    print(
        "Previous deck_count     :",
        previous_deck_count,
    )
    print(
        "Current  deck_count     :",
        deck_count,
    )

    if (
        previous_silhouette_score is not None
        and current_silhouette_score
        < previous_silhouette_score - 0.1
    ):
        print(
            "WARNING: silhouette score "
            "regressed by more than 0.1 "
            "compared to the previous run. "
            "Review the clustering result "
            "before trusting archetype_name "
            "assignments."
        )

In [0]:
print(
    "All archetype validations passed for "
    f"model={current_model_name}, "
    f"cluster_count={current_cluster_count}, "
    f"silhouette_score="
    f"{current_silhouette_score}, "
    f"deck_count={deck_count}"
)